# ES 335: Anomaly models (known counties) and leave-states-out (unseen states)
Needs `corn_model_table.csv` and `corn_yield.csv` in your Drive folder. **2022 is never used here.**

- **Setting A:** each county's own history is known; predict a future year as county mean + a weather-driven correction.
- **Setting B:** whole states are held out; no county history, so direct models compete with a training-average baseline.

In [ ]:
import os, re, warnings
import numpy as np
import pandas as pd
from google.colab import drive
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
warnings.filterwarnings("ignore")

drive.mount("/content/drive")
DATA_DIR = "/content/drive/MyDrive/es335_crop"
df = pd.read_csv(f"{DATA_DIR}/corn_model_table.csv", dtype={"FIPS": str, "STATEFP": str})
df = df[df["Year"] <= 2021].copy()                 # 2022 stays untouched
print(df.shape, sorted(df["Year"].unique()))

Mounted at /content/drive
(7176, 62) [np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021)]


In [ ]:
weather_cols = [c for c in df.columns if re.match(r"(tavg|tmax|tmin|precip|rh|vpd|srad|heat_days)_", c)]
soil_cols    = [c for c in df.columns if c.startswith(("nitrogen", "soc", "phh2o"))]
geo_cols     = ["lat", "lon"]

def scores(y, p):
    return {"RMSE": np.sqrt(mean_squared_error(y, p)), "MAE": mean_absolute_error(y, p), "R2": r2_score(y, p)}

## Setting A: known counties, predict 2021 from 2017-2020

In [ ]:
train = df[df["Year"].between(2017, 2020)].copy()
val   = df[df["Year"] == 2021].copy()

# --- leave-one-year-out county means for TRAINING rows (a row's own year is excluded) ---
cols = ["Yield"] + weather_cols
x    = train[cols]
sums = train.groupby("FIPS")[cols].transform("sum")
cnt  = train.groupby("FIPS")[cols].transform("count")
loo_mean = (sums - x.fillna(0)) / (cnt - x.notna().astype(int))
loo_mean = loo_mean.where((cnt - x.notna().astype(int)) >= 1)

has_hist = train.groupby("FIPS")["Yield"].transform("count") >= 2
train = train[has_hist].copy(); loo_mean = loo_mean[has_hist]

y_anom   = train["Yield"] - loo_mean["Yield"]
X_anom_w = train[weather_cols] - loo_mean[weather_cols]          # weather departures from the county's normal
print("training rows used:", len(train))

# --- validation rows: county means from ALL training years ---
cm_all = df[df["Year"].between(2017, 2020)].groupby("FIPS")[["Yield"] + weather_cols].mean()
known  = val["FIPS"].isin(cm_all.index)
print("2021 rows:", len(val), "| with county history:", known.sum(), "| without:", (~known).sum())

cm_val = cm_all.reindex(val["FIPS"])
val_anom_w = val[weather_cols].values - cm_val[weather_cols].values
val_anom_w = pd.DataFrame(val_anom_w, columns=weather_cols, index=val.index)

# fallback for counties with no history: state mean of training yields
state_mean = df[df["Year"].between(2017, 2020)].groupby("STATEFP")["Yield"].mean()
fallback = val["STATEFP"].map(state_mean).fillna(df["Yield"].mean())
base_pred = pd.Series(cm_val["Yield"].values, index=val.index).fillna(fallback)
base_rmse = scores(val["Yield"], base_pred)["RMSE"]
print("county-mean baseline:", {k: round(v, 2) for k, v in scores(val["Yield"], base_pred).items()})

training rows used: 5500
2021 rows: 1469 | with county history: 1430 | without: 39
county-mean baseline: {'RMSE': np.float64(23.83), 'MAE': 18.77, 'R2': 0.61}


In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor

def make_models():
    return {
      "ridge": make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), Ridge(alpha=100)),
      "rf":    make_pipeline(SimpleImputer(strategy="median"),
                             RandomForestRegressor(n_estimators=300, min_samples_leaf=10, n_jobs=-1, random_state=42)),
      "hgb":   HistGradientBoostingRegressor(max_iter=200, learning_rate=0.05, max_depth=4,
                                             min_samples_leaf=40, random_state=42),
    }

def build(fs):
    Xtr = X_anom_w.copy(); Xva = val_anom_w.copy()
    if "geo" in fs:  Xtr[geo_cols]  = train[geo_cols];  Xva[geo_cols]  = val[geo_cols]
    if "soil" in fs: Xtr[soil_cols] = train[soil_cols]; Xva[soil_cols] = val[soil_cols]
    return Xtr, Xva

rows = []
for fs in [("weather",), ("weather", "geo"), ("weather", "geo", "soil")]:
    Xtr, Xva = build(fs)
    for name, m in make_models().items():
        m.fit(Xtr, y_anom)
        corr = pd.Series(m.predict(Xva), index=val.index)
        corr[~known.values] = 0.0                      # no county history: fall back to the baseline
        pred = base_pred + corr
        rows.append({"features": "+".join(fs), "model": name, **scores(val["Yield"], pred)})

resA = pd.DataFrame(rows)
resA["RMSE_gain_vs_baseline_%"] = ((base_rmse - resA["RMSE"]) / base_rmse * 100).round(1)
print("baseline RMSE:", round(base_rmse, 2))
print(resA.round(2).to_string(index=False))
resA.to_csv(f"{DATA_DIR}/settingA_anomaly_val2021.csv", index=False)

baseline RMSE: 23.83
        features model  RMSE   MAE   R2  RMSE_gain_vs_baseline_%
         weather ridge 21.22 15.92 0.69                     10.9
         weather    rf 20.66 15.71 0.71                     13.3
         weather   hgb 20.40 15.39 0.71                     14.4
     weather+geo ridge 21.22 15.92 0.69                     10.9
     weather+geo    rf 20.45 15.46 0.71                     14.2
     weather+geo   hgb 20.47 15.45 0.71                     14.1
weather+geo+soil ridge 21.22 15.92 0.69                     10.9
weather+geo+soil    rf 20.48 15.48 0.71                     14.0
weather+geo+soil   hgb 20.38 15.42 0.71                     14.5


## Setting B: leave-states-out (2017-2021 data, 5 folds grouped by state)
Whole states are held out, so no county history exists. The baseline is the average yield of the training states.

In [ ]:
from sklearn.model_selection import GroupKFold

gkf = GroupKFold(n_splits=min(5, df["STATEFP"].nunique()))
groups = df["STATEFP"]
sets = {"weather":          weather_cols,
        "weather+geo":      weather_cols + geo_cols,
        "weather+geo+soil": weather_cols + geo_cols + soil_cols,
        "geo+soil (no weather)": geo_cols + soil_cols}

oof = {(fs, mname): np.zeros(len(df)) for fs in sets for mname in ["rf", "hgb"]}
base_oof = np.zeros(len(df))
for tr, te in gkf.split(df, groups=groups):
    base_oof[te] = df.iloc[tr]["Yield"].mean()
    for fs, cols in sets.items():
        for mname in ["rf", "hgb"]:
            m = make_models()[mname].fit(df.iloc[tr][cols], df.iloc[tr]["Yield"])
            oof[(fs, mname)][te] = m.predict(df.iloc[te][cols])

y = df["Yield"].values
b = scores(y, base_oof); print("training-mean baseline:", {k: round(v, 2) for k, v in b.items()})
rows = [{"features": fs, "model": mname, **scores(y, p)} for (fs, mname), p in oof.items()]
resB = pd.DataFrame(rows)
resB["RMSE_gain_vs_baseline_%"] = ((b["RMSE"] - resB["RMSE"]) / b["RMSE"] * 100).round(1)
print(resB.round(2).to_string(index=False))
resB.to_csv(f"{DATA_DIR}/settingB_leave_states_out.csv", index=False)

# which states are hardest?
err = pd.DataFrame({"STATEFP": df["STATEFP"], "se": (y - oof[("weather+geo+soil", "hgb")]) ** 2})
print(err.groupby("STATEFP")["se"].mean().pow(0.5).sort_values(ascending=False).head(8).round(1))

training-mean baseline: {'RMSE': np.float64(36.49), 'MAE': 29.11, 'R2': -0.01}
             features model  RMSE   MAE   R2  RMSE_gain_vs_baseline_%
              weather    rf 29.18 21.97 0.35                     20.0
              weather   hgb 29.19 21.88 0.35                     20.0
          weather+geo    rf 28.89 21.47 0.37                     20.8
          weather+geo   hgb 28.56 21.31 0.38                     21.7
     weather+geo+soil    rf 29.58 21.87 0.33                     18.9
     weather+geo+soil   hgb 29.51 21.87 0.34                     19.1
geo+soil (no weather)    rf 33.20 24.71 0.16                      9.0
geo+soil (no weather)   hgb 32.94 24.53 0.17                      9.7
STATEFP
53    124.9
16     85.3
08     64.2
13     47.0
48     46.6
31     40.4
46     40.0
20     37.6
Name: se, dtype: float64
